# 02b — As 11 Perguntas de Negócio (Parte 2)

> **Etapa 3B do Datathon FIAP Fase 5 — Associação Passos Mágicos**

## Esta entrega cobre as perguntas **7 a 11** — análises estratégicas e insights livres:

| # | Pergunta | Tipo |
|---|---|---|
| **P7** | IPV — feature importance preliminar | Prepara terreno pro modelo |
| **P8** | Multidimensionalidade — combinações que elevam o INDE | Estrutural |
| **P9** | Efetividade — jornada Quartzo → Topázio funciona? | Estratégico |
| **P10** | 🔥 Análise de evasão — quem sai do programa? | **Insight livre** |
| **P11** | 🔥 A crise da fase 3 + vácuo do ensino superior | **Insight livre** |

As perguntas 10 e 11 são os **grandes achados livres** do projeto — insights emergentes dos dados que não estavam no briefing, mas que viraram estratégicos depois da análise.

> ⚠️ **Este notebook assume que você já rodou `01_eda_limpeza.ipynb` e `02_perguntas_negocio.ipynb`.**

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

PROJECT_ROOT = Path.cwd().parent
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

alunos = pd.read_parquet(DATA_PROCESSED / "alunos_long.parquet")
print(f"✅ Dataset carregado: {alunos.shape}")

# Paletas consistentes
CORES_ANO = {2022: "#66c2a5", 2023: "#fc8d62", 2024: "#8da0cb"}
CORES_PEDRA = {"Quartzo": "#e63946", "Ágata": "#f4a261",
               "Ametista": "#2a9d8f", "Topázio": "#264653"}
ORDEM_PEDRAS = ["Quartzo", "Ágata", "Ametista", "Topázio"]

def salvar_figura(nome):
    path = FIGURES_DIR / f"{nome}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  💾 {path.relative_to(PROJECT_ROOT)}")

---

## P7 — O que mais influencia o Ponto de Virada (IPV)?

**Contexto:** O IPV mede o "ponto de virada" — momento em que o aluno dá um salto qualitativo no desenvolvimento. Pra Passos Mágicos, é **o indicador mais estratégico**, porque representa transformação real. Vamos descobrir quais indicadores mais influenciam o IPV, em dois olhares complementares:

1. **Contemporâneo** (indicadores em t correlacionam com IPV em t)
2. **Forward-looking** (indicadores em t correlacionam com IPV em t+1) — esse é o preview de **quais features vão entrar no modelo preditivo da Etapa 4**

In [ ]:
# Correlações com IPV por ano (contemporâneas)
preditores = ["ian", "ida", "ieg", "iaa", "ips", "ipp"]

corrs_contemporaneo = {}
for ano in [2022, 2023, 2024]:
    df_ano = alunos[alunos.ano == ano]
    row = {}
    for p in preditores:
        if df_ano[p].notna().sum() > 100:
            row[p.upper()] = df_ano[[p, "ipv"]].corr().iloc[0, 1]
        else:
            row[p.upper()] = np.nan
    corrs_contemporaneo[ano] = row

corr_df = pd.DataFrame(corrs_contemporaneo).round(3)
print("Correlação contemporânea com IPV (por ano):")
print(corr_df.to_string())

In [ ]:
# Visualização: evolução das correlações com IPV ao longo do tempo
fig, ax = plt.subplots(figsize=(11, 5))
for ind in corr_df.index:
    vals = corr_df.loc[ind].values
    ax.plot([2022, 2023, 2024], vals, marker="o", linewidth=2,
            markersize=8, label=ind)

ax.set_title("Correlação dos indicadores com IPV — evolução temporal", fontweight="bold")
ax.set_xlabel("Ano")
ax.set_ylabel("Correlação com IPV")
ax.set_xticks([2022, 2023, 2024])
ax.axhline(0, color="black", linewidth=0.5)
ax.legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.25))
ax.grid(True, alpha=0.3)
plt.tight_layout()
salvar_figura("p7_ipv_correlacoes_temporal")
plt.show()

In [ ]:
# Análise forward-looking: indicadores em t prevendo IPV em t+1
# Este é o PREVIEW do modelo preditivo
def construir_pares(ano_t, ano_tp1):
    t = alunos[alunos.ano == ano_t][["ra"] + preditores + ["ipv"]].rename(
        columns={c: c + "_t" for c in preditores + ["ipv"]})
    tp = alunos[alunos.ano == ano_tp1][["ra", "ipv"]].rename(
        columns={"ipv": "ipv_tp1"})
    return t.merge(tp, on="ra", how="inner").dropna()

pares = construir_pares(2023, 2024)
print(f"Pares 2023→2024 completos: {len(pares)}")

# Correlações forward-looking
corrs_forward = {}
for p in preditores + ["ipv"]:
    col = p + "_t"
    if col in pares.columns:
        corrs_forward[p.upper()] = pares[[col, "ipv_tp1"]].corr().iloc[0, 1]

corrs_forward_series = pd.Series(corrs_forward).sort_values(ascending=False)
print("\nCorrelação de indicadores(t) com IPV(t+1):")
for ind, v in corrs_forward_series.items():
    bar = "█" * int(v * 30)
    print(f"  {ind:5s}(t): {v:+.3f}  {bar}")

### 📊 Leitura quantitativa

**Correlações contemporâneas com IPV:**

| Indicador | 2022 | 2023 | 2024 |
|---|---|---|---|
| IDA | 0.62 | 0.54 | 0.51 |
| IEG | 0.59 | 0.45 | 0.54 |
| **IPP** | ❌ | 0.52 | **0.75** ⬆️ |
| IAA | 0.26 | 0.14 | 0.15 |
| IPS | 0.21 | 0.08 | 0.10 |
| IAN | 0.11 | 0.15 | 0.18 |

**Forward-looking (2023→2024):** IPV(t) = 0.47, IDA(t) = 0.42, IPP(t) = 0.41, IEG(t) = 0.36, IAN(t) = 0.32, IAA(t) = 0.15, IPS(t) = 0.08

### 🔍 Interpretação

- **IPP saltou de 0.52 em 2023 pra 0.75 em 2024** como preditor do IPV. Isso é **muito forte** — sugere que a equipe psicopedagógica ficou muito mais calibrada em identificar quem vai dar o salto qualitativo.
- **IDA e IEG** são os preditores mais estáveis e consistentes. Não tem volta — acadêmico + engajamento = virada.
- **IPS contribui quase nada** (corroborando P5). Vamos entrar na Etapa 4 sabendo que IPS **não deveria** ter peso grande no modelo, ou se tiver, precisa ser normalizado por ano.
- **Forward-looking**: o próprio IPV passado é o melhor preditor — quem já virou tende a se manter. Mas IDA(t) e IPP(t) vêm logo atrás.

### 💡 Implicações pro modelo preditivo (Etapa 4)

> **Features de maior peso esperadas:** IPV(t), IDA(t), IPP(t), IEG(t), IAN(t), delta_IDA, delta_IEG
>
> **Features de peso baixo/nulo esperadas:** IPS(t), IAA(t) — a autoavaliação é ruído puro (vimos na P4) e o IPS tem problemas metodológicos.
>
> **Recomendação executiva:** a Passos pode simplificar o PEDE. Se IAA e IPS não predizem nem desempenho futuro nem virada, vale a pena questionar o custo/benefício de mantê-los como estão.

---

## P8 — Quais combinações de indicadores mais elevam o INDE?

**Contexto:** O INDE é a métrica-mãe do PEDE. Mas ele **é uma combinação linear dos outros 7 indicadores** (pela construção oficial da ONG). A pergunta que faz sentido aqui é: **qual o peso real de cada indicador no INDE?** E: **existem alunos com INDE alto mas com perfis "desbalanceados"?**

Vamos responder as duas usando regressão linear pra descobrir os pesos empíricos, e depois analisando os perfis do top 25%.

In [ ]:
from sklearn.linear_model import LinearRegression

# Regressão linear: INDE ~ todos os indicadores
df_completo = alunos.dropna(subset=["inde"] + preditores + ["ipv"])
print(f"Alunos com todos indicadores + INDE: {len(df_completo)}")
print("(só 2023 e 2024, porque IPP não existe em 2022)")

X = df_completo[preditores + ["ipv"]]
y = df_completo["inde"]

lr = LinearRegression()
lr.fit(X, y)

print(f"\nR² da regressão: {lr.score(X, y):.4f}")
print("(esperamos próximo de 1.0 porque INDE é combinação linear dos 7 por construção)")

# Pesos normalizados (% de contribuição)
pesos = pd.Series(lr.coef_, index=X.columns)
pesos_norm = (pesos / pesos.sum() * 100).round(2).sort_values(ascending=False)
print("\nPesos empíricos no INDE:")
for ind, peso in pesos_norm.items():
    bar = "█" * int(peso)
    print(f"  {ind.upper():5s}: {peso:5.1f}%  {bar}")

In [ ]:
# Visualização: pesos descobertos
fig, ax = plt.subplots(figsize=(10, 5))
cores_pesos = ["#264653" if v >= 15 else "#2a9d8f" if v >= 12 else "#e9c46a"
               for v in pesos_norm.values]
bars = ax.barh(pesos_norm.index.astype(str).str.upper(),
               pesos_norm.values, color=cores_pesos)
ax.set_title("Pesos empíricos dos indicadores no INDE\n"
             "(descobertos via regressão linear)", fontweight="bold")
ax.set_xlabel("Contribuição relativa no INDE (%)")
ax.axvline(100 / 7, color="red", linestyle="--", alpha=0.5,
           label=f"Peso uniforme ({100/7:.1f}%)")
for bar, v in zip(bars, pesos_norm.values):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2,
            f"{v:.1f}%", va="center", fontweight="bold")
ax.legend()
plt.tight_layout()
salvar_figura("p8_pesos_inde")
plt.show()

In [ ]:
# Análise dos perfis do top 25% de INDE
p75 = df_completo["inde"].quantile(0.75)
top = df_completo[df_completo["inde"] >= p75].copy()
print(f"Top 25% do INDE: >= {p75:.2f} ({len(top)} alunos)")

# Quantos indicadores "altos" (>=7) cada aluno top25 tem?
for ind in ["ida", "ieg", "iaa", "ips", "ipp", "ipv"]:
    top[f"{ind}_alto"] = (top[ind] >= 7).astype(int)

top["n_altos"] = top[[f"{i}_alto" for i in ["ida", "ieg", "iaa", "ips", "ipp", "ipv"]]].sum(axis=1)

print("\nDistribuição de nº de indicadores 'altos' (≥7) dentro do top25:")
for n, count in top["n_altos"].value_counts().sort_index().items():
    pct = count / len(top) * 100
    bar = "█" * int(pct)
    print(f"  {int(n)}/6 altos: {count:3d} ({pct:.1f}%) {bar}")

# Combinações IDA × IEG × IPV
print("\n\nPerfis do tripé IDA × IEG × IPV (A=alto ≥7, B=baixo <7) dentro do top25:")
perfis = top.groupby(["ida_alto", "ieg_alto", "ipv_alto"]).size().sort_values(ascending=False)
for (ida_a, ieg_a, ipv_a), n in perfis.items():
    perfil = f"IDA={'A' if ida_a else 'B'} IEG={'A' if ieg_a else 'B'} IPV={'A' if ipv_a else 'B'}"
    pct = n / len(top) * 100
    print(f"  {perfil}: {n:3d} ({pct:.1f}%)")

### 📊 Leitura quantitativa

**Pesos empíricos descobertos:**

| Indicador | Peso |
|---|---|
| **IEG, IDA, IPV** | **~20% cada** |
| IAN, IAA, IPS, IPP | ~10% cada |

Isso reproduz com precisão a fórmula oficial do PEDE — a ONG dá peso dobrado ao tripé **desempenho × engajamento × virada**.

**Perfis do top 25% (497 alunos):**

- **90.5%** dos alunos do top25 têm o tripé IDA + IEG + IPV todos altos
- **Só 43 alunos (8.7%)** chegam ao top sem IDA alto (têm engajamento + virada emocional, mas não performance acadêmica)
- **Só 4 alunos (0.8%)** chegam ao top sem IPV alto (tem desempenho e engajamento mas sem "virada qualitativa")
- **263 alunos (52.9%)** têm 6/6 indicadores altos — são os alunos "completos"

### 🔍 Interpretação

**Não existe "atalho" pra chegar ao topo do INDE.** Quase ninguém consegue ser top-tier sem os 3 grandes componentes do tripé (IDA + IEG + IPV). Isso é bom — mostra que o PEDE está pesando os fatores certos.

Os **43 alunos "engajados sem performance"** são interessantes: eles têm engajamento + virada emocional mas **desempenho acadêmico fraco**. Podem ser casos onde o aluno está *super motivado* mas com dificuldade cognitiva ou de base. São candidatos a **intervenção acadêmica intensiva** — o engajamento tá lá, precisa só destravar o aprendizado.

### 💡 O que isso significa pra Passos Mágicos

> **A fórmula do INDE está bem calibrada — não há o que mexer na mecânica.** O problema não é a métrica, é a *composição do grupo*: a maioria dos alunos não chega no top porque tem pelo menos um dos 3 pilares fraco, e os 3 são necessários.

> **Recomendação executiva:** ao invés de tentar "levantar INDE" genericamente, focar em **identificar qual é o pilar mais fraco de cada aluno** e personalizar intervenção. O top25 mostra que quem sobe, sobe nos 3 pilares ao mesmo tempo.

---

## P9 — A jornada Quartzo → Topázio funciona? O programa é efetivo?

**Contexto:** As pedras (Quartzo → Ágata → Ametista → Topázio) representam níveis de desenvolvimento. A ideia é que, com o tempo no programa, o aluno evolua. Mas isso **realmente acontece**? E se sim, **em que velocidade**?

Essa é a pergunta que responde "o programa funciona?" de forma direta, olhando pras **transições efetivas** dos alunos entre anos.

In [ ]:
ORDEM = {"Quartzo": 0, "Ágata": 1, "Ametista": 2, "Topázio": 3}

def matriz_transicao(ano_t, ano_tp1):
    t = alunos[alunos.ano == ano_t][["ra", "pedra"]].dropna()
    tp = alunos[alunos.ano == ano_tp1][["ra", "pedra"]].dropna()
    m = t.merge(tp, on="ra", suffixes=("_t", "_tp1"))
    return m

tr_22_23 = matriz_transicao(2022, 2023)
tr_23_24 = matriz_transicao(2023, 2024)

def resumo(tr, label):
    tr = tr.copy()
    tr["nivel_t"] = tr["pedra_t"].map(ORDEM)
    tr["nivel_tp1"] = tr["pedra_tp1"].map(ORDEM)
    tr["delta"] = tr["nivel_tp1"] - tr["nivel_t"]
    melhor = (tr["delta"] > 0).sum()
    estav = (tr["delta"] == 0).sum()
    pior = (tr["delta"] < 0).sum()
    print(f"\n{label} (n={len(tr)}):")
    print(f"  Melhorou: {melhor} ({melhor/len(tr)*100:.1f}%)")
    print(f"  Estável:  {estav} ({estav/len(tr)*100:.1f}%)")
    print(f"  Piorou:   {pior} ({pior/len(tr)*100:.1f}%)")

resumo(tr_22_23, "2022 → 2023")
resumo(tr_23_24, "2023 → 2024")

In [ ]:
# Visualização: 2 heatmaps de matriz de transição
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, tr, titulo in zip(axes, [tr_22_23, tr_23_24], ["2022 → 2023", "2023 → 2024"]):
    matriz = pd.crosstab(tr["pedra_t"], tr["pedra_tp1"])
    matriz = matriz.reindex(index=ORDEM_PEDRAS, columns=ORDEM_PEDRAS, fill_value=0)
    # Normalizar por linha pra ver a probabilidade de transição
    matriz_pct = matriz.div(matriz.sum(axis=1), axis=0) * 100
    sns.heatmap(matriz_pct, annot=True, fmt=".1f", cmap="YlGn",
                cbar_kws={"label": "% da linha"},
                linewidths=0.5, ax=ax, vmin=0, vmax=100)
    ax.set_title(f"Transição de pedras — {titulo}\n(% por linha)", fontweight="bold")
    ax.set_xlabel("Pedra no ano seguinte")
    ax.set_ylabel("Pedra no ano atual")

plt.tight_layout()
salvar_figura("p9_matriz_transicoes")
plt.show()

In [ ]:
# Jornada dos Quartzo de 2022 — os mais frágeis
quartzo_22 = set(alunos[(alunos.ano == 2022) & (alunos.pedra == "Quartzo")].ra)
print(f"Alunos Quartzo em 2022: {len(quartzo_22)}")

# Quantos sobreviveram até 2024?
quartzo_em_24 = alunos[(alunos.ra.isin(quartzo_22)) & (alunos.ano == 2024) & (alunos.pedra.notna())]
print(f"Desses, ainda com pedra atribuída em 2024: {len(quartzo_em_24)}")
print(f"Taxa de evasão: {(len(quartzo_22) - len(quartzo_em_24)) / len(quartzo_22) * 100:.1f}%")

print("\nDistribuição em 2024 dos 'Quartzo de 2022 que ficaram':")
print(quartzo_em_24["pedra"].value_counts())

# Comparar com Topázio de 2022
topazio_22 = set(alunos[(alunos.ano == 2022) & (alunos.pedra == "Topázio")].ra)
topazio_em_24 = alunos[(alunos.ra.isin(topazio_22)) & (alunos.ano == 2024) & (alunos.pedra.notna())]
print(f"\n\nAlunos Topázio em 2022: {len(topazio_22)}")
print(f"Desses, ainda com pedra em 2024: {len(topazio_em_24)}")
print(f"Taxa de evasão: {(len(topazio_22) - len(topazio_em_24)) / len(topazio_22) * 100:.1f}%")
print("\nDistribuição em 2024 dos 'Topázio de 2022 que ficaram':")
print(topazio_em_24["pedra"].value_counts())

### 📊 Leitura quantitativa

**Distribuição das transições (~25% melhora / ~50% estável / ~25% piora, nos dois anos)** — simétrica, próxima da aleatoriedade.

**Jornada dos extremos (2022 → 2024):**

| | Quartzo (mais frágil) | Topázio (topo) |
|---|---|---|
| Alunos em 2022 | 132 | 130 |
| Evadiram até 2024 | **78 (59%)** | **12 (9%)** |
| Ficaram | 54 | 118 |
| Destino dos que ficaram | 8 Quartzo, 11 Ágata, 7 Ametista, 0 Topázio | 64 Topázio, 25 Ametista |

### 🔍 Interpretação

**A simetria das transições (25/50/25) é preocupante.** Se o programa tivesse um efeito causal forte de empurrar alunos pra cima, esperaríamos uma distribuição **assimétrica** (mais melhora do que piora). O que a gente vê é quase um passeio aleatório.

**Mas o dado mais duro é a evasão por pedra:**

- **59% dos Quartzo de 2022 evadiram** até 2024
- **9% dos Topázio de 2022** evadiram

**6× mais evasão nos frágeis.** O programa retém quem já está bem e perde quem mais precisa. Isso **desloca a média** pra cima artificialmente: quando vc olha o "INDE médio subindo", parte desse ganho é porque os mais fracos saíram da amostra.

Dos 54 Quartzo que aguentaram 2 anos, **apenas 7 chegaram a Ametista. ZERO chegaram a Topázio.** A jornada Quartzo→Topázio em 2 anos é **praticamente impossível** na prática — teria que pular 3 níveis.

### 💡 O que isso significa pra Passos Mágicos

> ⚠️ **O programa é melhor em manter quem já está bem do que em soerguer quem está mal.** Isso é uma verdade incômoda mas importante pra apresentação executiva. Não é culpa da ONG — é característica conhecida de intervenção educacional, especialmente em contextos de vulnerabilidade social.

> **Recomendação executiva:**
> 1. **Aceitar que a jornada Quartzo → Topázio leva 3+ anos, não 2.** Definir expectativas realistas com doadores e equipe.
> 2. **Focar retenção dos Quartzo** como KPI prioritário. Reduzir evasão de 59% pra 40% salvaria dezenas de alunos por ano.
> 3. **Desenhar intervenção específica pra Quartzo** — eles não precisam do mesmo programa dos Ametista. O programa atual é "one-size-fits-all" e está deixando os mais frágeis pra trás.

---

## P10 — 🔥 INSIGHT LIVRE: Quem evade do programa?

**Contexto:** Durante a Etapa 2 a gente descobriu que **256 alunos presentes em 2022 simplesmente desapareceram** — nunca mais apareceram em 2023 ou 2024. Isso é **29.8% da base de 2022**. Quem são esses alunos? Era previsível que iam sair? **Se a gente tivesse um modelo preditivo bom, conseguiríamos ter salvado eles?**

Essa análise é **o coração do case** pra ONG: reduzir evasão é a alavanca mais direta de impacto.

In [ ]:
# Identificar evadidos: presentes em 2022, ausentes em 2023 E 2024
ra_por_ano = {ano: set(alunos[alunos.ano == ano].ra) for ano in [2022, 2023, 2024]}
evadidos_22 = ra_por_ano[2022] - ra_por_ano[2023] - ra_por_ano[2024]

# Criar flag no dataset
alunos_22 = alunos[alunos.ano == 2022].copy()
alunos_22["evadiu"] = alunos_22.ra.isin(evadidos_22)

print(f"Alunos em 2022: {len(alunos_22)}")
print(f"Evadiram: {alunos_22['evadiu'].sum()} ({alunos_22['evadiu'].mean() * 100:.1f}%)")
print(f"Retidos: {(~alunos_22['evadiu']).sum()} ({(~alunos_22['evadiu']).mean() * 100:.1f}%)")

In [ ]:
# Teste estatístico: quais indicadores diferem entre evadidos e retidos?
indicadores = ["inde", "ian", "ida", "ieg", "iaa", "ips", "ipv"]

print("Comparação de indicadores (Retidos vs. Evadidos):")
print(f"{'Indicador':<10} {'Retidos':>10} {'Evadiram':>10} {'Delta':>8} {'p-valor':>10}")
print("-" * 55)

resultados_t = []
for ind in indicadores:
    retidos = alunos_22[~alunos_22["evadiu"]][ind].dropna()
    evadidos_v = alunos_22[alunos_22["evadiu"]][ind].dropna()
    if len(evadidos_v) > 10 and len(retidos) > 10:
        t, p = stats.ttest_ind(retidos, evadidos_v)
        delta = retidos.mean() - evadidos_v.mean()
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"{ind.upper():<10} {retidos.mean():>10.2f} {evadidos_v.mean():>10.2f} "
              f"{delta:>+8.2f} {p:>10.4f} {sig}")
        resultados_t.append({"indicador": ind.upper(), "delta": delta, "p": p})

resultados_df = pd.DataFrame(resultados_t)

In [ ]:
# Visualização: comparar distribuições de indicadores entre evadidos e retidos
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
indicadores_plot = ["inde", "ida", "ieg", "iaa", "ian", "ips", "ipv"]

for ax, ind in zip(axes.flat, indicadores_plot):
    for cor, label, mask in [("#2a9d8f", "Retidos", ~alunos_22["evadiu"]),
                              ("#e63946", "Evadiram", alunos_22["evadiu"])]:
        dados = alunos_22[mask][ind].dropna()
        if len(dados) > 5:
            sns.kdeplot(dados, ax=ax, color=cor, label=label, linewidth=2, fill=True, alpha=0.2)
    ax.set_title(ind.upper(), fontweight="bold")
    ax.set_xlabel("")
    ax.set_xlim(0, 10)
    ax.legend(fontsize=8)

axes.flat[-1].axis("off")  # último subplot fica vazio
plt.suptitle("Distribuição de indicadores: Retidos vs. Evadiram (base 2022)",
             fontsize=14, y=1.02, fontweight="bold")
plt.tight_layout()
salvar_figura("p10_distribuicoes_evasao")
plt.show()

In [ ]:
# Taxa de evasão por PEDRA
taxa_pedra = (alunos_22.dropna(subset=["pedra"])
              .groupby("pedra")["evadiu"].agg(["sum", "count", "mean"]))
taxa_pedra.columns = ["evadiram", "total", "taxa_evasao"]
taxa_pedra["taxa_evasao"] = (taxa_pedra["taxa_evasao"] * 100).round(1)
taxa_pedra = taxa_pedra.reindex(ORDEM_PEDRAS)
print("Taxa de evasão por pedra:")
print(taxa_pedra)

# Visualização
fig, ax = plt.subplots(figsize=(10, 5))
cores_pedras_lista = [CORES_PEDRA[p] for p in ORDEM_PEDRAS]
bars = ax.bar(ORDEM_PEDRAS, taxa_pedra["taxa_evasao"], color=cores_pedras_lista)
ax.set_title("Taxa de evasão por pedra (alunos presentes em 2022 que nunca voltaram)",
             fontweight="bold")
ax.set_xlabel("Pedra em 2022")
ax.set_ylabel("% que evadiu")
ax.set_ylim(0, 70)
ax.axhline(29.8, color="gray", linestyle="--", label="Taxa média (29.8%)")
for bar, v, n in zip(bars, taxa_pedra["taxa_evasao"], taxa_pedra["total"]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1.5,
            f"{v:.1f}%\n({int(n)} alunos)", ha="center", fontweight="bold")
ax.legend()
plt.tight_layout()
salvar_figura("p10_evasao_por_pedra")
plt.show()

In [ ]:
# Taxa de evasão por fase
taxa_fase = (alunos_22.dropna(subset=["fase"])
             .groupby("fase")
             .agg(evadiram=("evadiu", "sum"),
                  total=("evadiu", "count"),
                  taxa=("evadiu", "mean")))
taxa_fase = taxa_fase[taxa_fase["total"] >= 10]
taxa_fase["taxa_pct"] = (taxa_fase["taxa"] * 100).round(1)
print("Taxa de evasão por fase (mínimo 10 alunos):")
print(taxa_fase[["evadiram", "total", "taxa_pct"]])

# Visualização
fig, ax = plt.subplots(figsize=(11, 5))
colors_fase = ["#e63946" if v >= 40 else "#f4a261" if v >= 30 else "#2a9d8f"
               for v in taxa_fase["taxa_pct"]]
bars = ax.bar(taxa_fase.index.astype(int).astype(str),
              taxa_fase["taxa_pct"], color=colors_fase)
ax.set_title("Taxa de evasão por fase (base 2022)", fontweight="bold")
ax.set_xlabel("Fase")
ax.set_ylabel("% que evadiu")
ax.axhline(29.8, color="gray", linestyle="--", label="Taxa média (29.8%)")
for bar, v in zip(bars, taxa_fase["taxa_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1,
            f"{v:.1f}%", ha="center", fontweight="bold")
ax.legend()
plt.tight_layout()
salvar_figura("p10_evasao_por_fase")
plt.show()

### 📊 Leitura quantitativa

**Diferenças estatisticamente significativas entre evadidos e retidos:**

| Indicador | Delta (retido − evadido) | p-valor |
|---|---|---|
| **IEG** | **+1.32** | <0.001 *** |
| **IDA** | **+1.22** | <0.001 *** |
| **INDE** | +0.73 | <0.001 *** |
| **IPV** | +0.64 | <0.001 *** |
| IAA | +0.52 | <0.001 *** |
| IAN | +0.33 | ns (p=0.06) |
| **IPS** | **-0.02** | **ns (p=0.83)** |

**Taxa de evasão por pedra:**

- Quartzo: **59.1%** 🚨 (78 de 132)
- Ágata: 32.8% (82 de 250)
- Ametista: 24.1% (84 de 348)
- Topázio: 9.2% (12 de 130)

**Taxa de evasão por fase (destaques):**

- Fase 7 (3º EM): **47.6%** (10/21) 🚨
- Fase 3 (7º-8º ano): **41.9%** (62/148) 🚨
- Fase 0 (Alfa): 21.6% (41/190)

### 🔍 Interpretação

**Quem evade:** tem IEG e IDA significativamente menores, é Quartzo, está na fase 3 ou fase 7. **O perfil de evasão é previsível** — exatamente o que a gente precisa pro modelo preditivo da Etapa 4.

**IPS não prediz evasão** (p=0.83): outra confirmação de que o IPS como tá medido hoje **é ruído**.

**Fase 7 (3º ano EM)** tem evasão altíssima — mas interpretação é ambígua: pode ser evasão "positiva" (aluno entrou na faculdade e saiu do programa formal) ou "negativa" (desistiu perto do fim). **Precisa qualificar com dados adicionais** — mas a ONG sabe distinguir, é só perguntar.

**Fase 3** é o buraco real: 42% dos alunos da 7ª-8ª série desapareceram em 2 anos. **É a mesma fase com pior IDA**. A "crise da 8ª série" é a ferida mais grave do programa.

### 💡 O que isso significa pra Passos Mágicos

> 🎯 **Esta é a pergunta mais acionável do projeto inteiro.** A evasão é previsível pelos dados do PEDE, então **um modelo preditivo pode gerar impacto direto**: marcar os alunos de alto risco no início de cada ano e intervir antes da saída.

> **Recomendação executiva:**
> 1. **Programa anti-evasão específico pra Quartzo + Fase 3** — essa é a intersecção mais vulnerável. Se a gente reduzir a evasão desses 2 grupos em 25%, salva ~35 alunos/ano.
> 2. **O modelo preditivo da Etapa 4** vai usar IEG e IDA como features centrais, já que são os maiores diferenciadores.
> 3. **Entrevistar os evadidos** (se possível) — os dados dizem *quem* sai, mas não *por que*. Esse "por quê" é o que fecha o loop pra desenho de intervenção efetiva.

---

## P11 — 🔥 INSIGHT LIVRE: A crise da fase 3 e o vácuo do ensino superior

**Contexto:** Durante as análises anteriores, dois padrões apareceram como **pontos críticos não previstos no briefing**:

1. A **Fase 3** (7º-8º ano) é o buraco crônico do programa em todos os indicadores, todos os anos
2. O **ensino superior** (fases 8-9) é uma **expansão recente** da ONG, criada ad hoc em 2023-2024, que foge do instrumento PEDE

Essas duas coisas **não estavam no escopo original do datathon**, mas emergiram como **insights livres de alta importância estratégica**.

### 11.1 A crise da fase 3

In [ ]:
f3 = alunos[alunos["fase"] == 3].copy()
print(f"Alunos na fase 3 por ano:")
print(f3.groupby("ano").size())

# Médias da fase 3 vs global
print("\nIndicadores médios: Fase 3 vs média global")
print(f"{'Indicador':<10} {'Fase 3':>10} {'Global':>10} {'Delta':>10}")
print("-" * 44)
for ind in ["inde", "ida", "ieg", "iaa", "ips", "ipv"]:
    global_med = alunos[ind].mean()
    f3_med = f3[ind].mean()
    delta = f3_med - global_med
    marca = " 🚨" if delta < -0.5 else ""
    print(f"{ind.upper():<10} {f3_med:>10.2f} {global_med:>10.2f} {delta:>+10.2f}{marca}")

In [ ]:
# Visualização: radar da fase 3 vs global
indicadores = ["inde", "ida", "ieg", "iaa", "ips", "ipv"]
medias_globais = [alunos[i].mean() for i in indicadores]
medias_f3 = [f3[i].mean() for i in indicadores]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(indicadores))
width = 0.35

bars1 = ax.bar(x - width/2, medias_globais, width, label="Média global",
               color="#8da0cb", edgecolor="white", linewidth=1.5)
bars2 = ax.bar(x + width/2, medias_f3, width, label="Fase 3",
               color="#e63946", edgecolor="white", linewidth=1.5)

ax.set_xticks(x)
ax.set_xticklabels([i.upper() for i in indicadores])
ax.set_ylabel("Valor médio")
ax.set_title("Fase 3 vs média global — a ferida crônica do programa",
             fontweight="bold")
ax.set_ylim(0, 10)
ax.legend()

# Anotar deltas
for i, (g, f) in enumerate(zip(medias_globais, medias_f3)):
    delta = f - g
    cor_txt = "darkred" if delta < -0.3 else "darkgreen" if delta > 0 else "gray"
    ax.text(i, max(g, f) + 0.25, f"Δ={delta:+.2f}",
            ha="center", fontsize=9, fontweight="bold", color=cor_txt)

plt.tight_layout()
salvar_figura("p11_crise_fase3")
plt.show()

In [ ]:
# Taxa de sobrevivência específica da fase 3
ra_f3_22 = set(f3[f3.ano == 2022].ra)
ra_em_24 = ra_f3_22 & set(alunos[alunos.ano == 2024].ra)
print(f"Fase 3 em 2022: {len(ra_f3_22)} alunos")
print(f"Ainda no programa em 2024: {len(ra_em_24)} ({len(ra_em_24)/len(ra_f3_22)*100:.1f}%)")
print(f"Taxa de evasão: {(len(ra_f3_22)-len(ra_em_24))/len(ra_f3_22)*100:.1f}%")

### 🔍 Interpretação da Fase 3

A fase 3 é o **"vale da morte"** do programa:
- **IDA -0.98** abaixo da média (maior gap de todos)
- **IPV -0.55** abaixo
- **54.7% sobreviveram** até 2024 (taxa de evasão de 45%)
- **Única fase onde IPS está ACIMA da média** (+0.08) — o psicólogo vê eles bem, mas a realidade acadêmica diz o contrário

**Por que isso é estrutural:** a Fase 3 corresponde ao **7º-8º ano do ensino fundamental**. É a idade de 12-14 anos — entrada na adolescência, explosão hormonal, mudança abrupta da matemática concreta pra abstrata (álgebra), pressão social crescente. É um ponto de inflexão conhecido da educação brasileira em geral — não é específico da Passos.

Mas o dado mostra que **o programa atual não está preparado pra enfrentar isso**. Os indicadores psicossociais (IPS, IAA) não capturam a dificuldade — é um fenômeno **puramente acadêmico** (IDA) combinado com **desengajamento** (IEG). A resposta precisa ser **pedagógica específica**, não emocional.

### 11.2 O vácuo do ensino superior

In [ ]:
# Evolução da presença de alunos do ensino superior
print("Alunos no ensino superior (fases 8 e 9) por ano:")
for ano in [2022, 2023, 2024]:
    sub = alunos[alunos.ano == ano]
    f8 = (sub["fase"] == 8).sum()
    f9 = (sub["fase"] == 9).sum()
    print(f"  {ano}: {f8 + f9:4d} total  (fase 8: {f8}, fase 9: {f9})")

# Em 2024, quantos têm indicadores preenchidos?
sup_24 = alunos[(alunos.ano == 2024) & (alunos["fase"].isin([8, 9]))]
print(f"\nDos {len(sup_24)} universitários em 2024:")
for ind in ["inde", "pedra", "ida", "ieg", "ipv", "ipp"]:
    preenchidos = sup_24[ind].notna().sum()
    pct = preenchidos / len(sup_24) * 100
    print(f"  {ind.upper():6s}: {preenchidos:3d}/{len(sup_24)} ({pct:5.1f}%)")

In [ ]:
# Dos universitários em 2024, quantos vieram de ANTES (alunos da Passos históricos)
ra_sup_24 = set(sup_24["ra"])
ra_22 = set(alunos[alunos.ano == 2022].ra)
ra_23 = set(alunos[alunos.ano == 2023].ra)

eram_de_22 = ra_sup_24 & ra_22
eram_de_23 = ra_sup_24 & ra_23
novos_em_24 = ra_sup_24 - ra_22 - ra_23

print(f"Dos {len(ra_sup_24)} universitários em 2024:")
print(f"  Já estavam no programa em 2022: {len(eram_de_22)}")
print(f"  Entraram em 2023:               {len(eram_de_23 - ra_22)}")
print(f"  Totalmente novos em 2024:       {len(novos_em_24)}")

# Visualização: evolução do programa de ensino superior
fig, ax = plt.subplots(figsize=(10, 5))
anos = [2022, 2023, 2024]
fase_8 = [(alunos[alunos.ano == a]["fase"] == 8).sum() for a in anos]
fase_9 = [(alunos[alunos.ano == a]["fase"] == 9).sum() for a in anos]

ax.bar(anos, fase_8, label="Fase 8 (universitários)", color="#264653", width=0.6)
ax.bar(anos, fase_9, bottom=fase_8, label="Fase 9 (pós-fase-8)", color="#e9c46a", width=0.6)

for i, (a, f8, f9) in enumerate(zip(anos, fase_8, fase_9)):
    total = f8 + f9
    if total > 0:
        ax.text(a, total + 3, f"{total}", ha="center", fontweight="bold", fontsize=12)

ax.set_title("Expansão do programa de ensino superior — um vácuo em construção",
             fontweight="bold")
ax.set_xlabel("Ano")
ax.set_ylabel("Nº de alunos")
ax.set_xticks(anos)
ax.legend()
plt.tight_layout()
salvar_figura("p11_vacuo_ensino_superior")
plt.show()

### 🔍 Interpretação do vácuo do ensino superior

**Os números contam uma história clara:**

- **2022: 0 alunos** no ensino superior
- **2023: 63 alunos** (só fase 8)
- **2024: 102 alunos** (fase 8 + nova fase 9 criada)

**Dos 102 universitários em 2024:**

- **INDE preenchido**: 0 (0%) 🚨
- **Pedra atribuída**: 0 (0%) 🚨
- **IDA preenchido**: 1 (1%) 🚨
- **IEG preenchido**: 102 (100%) — só isso

A Passos Mágicos **criou um programa de continuidade universitária sem criar o instrumento de avaliação correspondente**. Ela está coletando algum sinal (IEG), mas nem de longe o suficiente pra medir impacto.

**Por que isso aconteceu:** porque o PEDE foi desenhado pra ensino fundamental e médio (pedras Quartzo-Topázio associadas a notas escolares, fase ideal associada a idade/série). Nada disso faz sentido pra universitário, então a ONG simplesmente **não aplica a avaliação**.

### 💡 O que isso significa pra Passos Mágicos

> **A Passos amadureceu junto com seus alunos.** A ONG começou com foco em crianças e agora está formando a primeira geração de universitários — um marco histórico. Mas está **voando cego** nesse novo pedaço do programa porque não tem instrumento de avaliação apropriado.

> **Recomendação executiva (a mais disruptiva do projeto):**
>
> A Passos precisa criar um **PEDE-Superior** — uma versão do instrumento adaptada pra ensino superior. Indicadores possíveis:
> - **Desempenho universitário**: CR, aprovação em disciplinas-chave
> - **Engajamento**: frequência em mentorias, participação em grupos de estudo
> - **Empregabilidade**: estágios conquistados, networking profissional
> - **Impacto de longo prazo**: egressos que retornam como mentores
>
> Sem isso, daqui a 5 anos a Passos não vai ter dados pra provar (pra doadores, pra sociedade, pra si mesma) que o programa universitário funcionou. **Essa é uma recomendação que vem dos dados, mas tem implicação estratégica imediata.**

---

## ✅ Resumo da Etapa 3B

### Os 5 grandes achados das perguntas 7-11

| # | Achado | Magnitude |
|---|---|---|
| **P7** | IDA, IEG, IPV e **IPP (crescendo)** são os preditores do IPV. IPS e IAA contribuem quase nada. | Features do modelo da Etapa 4 |
| **P8** | **Descobri a fórmula oficial do INDE** via regressão linear: IDA/IEG/IPV pesam 20% cada, os outros 4 pesam 10%. Sem atalho pro top25. | Validação estrutural |
| **P9** | Transições quase simétricas (25/50/25). **6× mais evasão no Quartzo**. O programa mantém quem já está bem. | Alerta estratégico |
| **P10** | 🔥 **Evasão é previsível**: IEG e IDA baixos, Quartzo, fase 3 ou 7. 59% dos Quartzo evadem. | **Justifica o modelo preditivo** |
| **P11** | 🔥 Fase 3 é o vale da morte (IDA -0.98). Ensino superior cresceu de 0 → 102 alunos em 2 anos, sem instrumento de avaliação. | **Recomendações disruptivas** |

### 🏆 Os 3 insights mais acionáveis do projeto inteiro (Parte 1 + Parte 2)

1. **P4 — Dunning-Kruger confirmado**: 65% dos alunos superestimam seu desempenho. Quem está pior é quem mais superestima. **Implicação pedagógica direta**: feedback explícito e frequente.

2. **P10 — Evasão previsível**: o modelo da Etapa 4 pode salvar dezenas de alunos/ano identificando risco antes da evasão acontecer. **Justificativa direta pro coração técnico do projeto.**

3. **P11 — Vácuo do ensino superior**: recomendação estratégica disruptiva — a Passos precisa criar um PEDE-Superior pra não perder a capacidade de medir seu próprio impacto no novo público.

### 🎯 Etapa 3 completa — o que temos agora
- ✅ 11 perguntas respondidas
- ✅ ~15 figuras prontas pra apresentação em `reports/figures/`
- ✅ Lista clara de features prioritárias pro modelo preditivo (IEG, IDA, IPV, IPP, IAN e deltas)
- ✅ Definição do target de risco com justificativa empírica forte
- ✅ Narrativa de negócio estruturada pro storytelling executivo

### Próximo passo
**Etapa 4 — Feature engineering e modelagem preditiva** (`03_feature_engineering.ipynb` e `04_modelagem.ipynb`).